# 04 · Avaliação da rodada

**O que esta etapa faz:** para cada carta da rodada, mede
- **acurácia** com as amostras de validação (20% que não entram no treino), por ano: acerto global, precisão, recall e F1 da classe urbana;
- **área urbana por ano**, comparada com a Coleção 11;
- **concordância pixel a pixel** com o mapa final da Coleção 11.

As amostras de validação vêm de áreas estáveis, então a acurácia tende a ser otimista nas áreas de transição. Use-a para **comparar rodadas**, não como acurácia absoluta.

As cartas avaliadas seguem `scope` e `cartas_teste` em `config/col12.yaml`.

## 0 · Preparação

In [ ]:
import sys, os
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    !git clone -q -b col12 https://github.com/jpedrassoli/brazil-urban-area.git repo 2>/dev/null || (cd repo && git pull -q)
    %pip install -q -e "repo/lulc_30m_landsat/collection 12" geemap
    RAIZ = 'repo/lulc_30m_landsat/collection 12'
else:
    RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

import ee, pandas as pd
from mb_urban import auth, config, grids, evaluate, col11
cfg = config.load()

## 1 · Conta do Earth Engine
Usa o login padrão do Earth Engine. No Colab, abre a autorização da sua conta.

In [ ]:
CONTA = 'default'
if CONTA == 'default':
    try:
        ee.Authenticate()   # no Colab abre uma janela de autorização: marque todas as permissões
    except Exception as erro:
        # se a janela falhar ("credential propagation was unsuccessful"), usa o login por link + código
        print('Autorização pela janela falhou; usando link e código:', erro)
        ee.Authenticate(auth_mode='notebook', force=True)
auth.init(CONTA)

## 2 · Cartas da rodada
O mapa avaliado de cada carta é `<destino>/<postprocess.folder>/mapa_<carta>`.

In [ ]:
CARTAS = grids.cartas_in_scope(cfg)
SUFIXO = ''   # sufixo do nome dos mapas, se a rodada usou um

# Para ver já o mapa da carta piloto (reprodução da Col 11), descomente as 3 linhas abaixo:
# cfg['postprocess']['folder'] = 'piloto_mapa_final'
# CARTAS = ['SF-23-Y-C']
# SUFIXO = '_reproducao_col11'

for c in CARTAS:
    print(c, '-', cfg['cartas_teste'].get(c, ''))

## 3 · Calcular as métricas
Pode levar alguns minutos por carta.

In [ ]:
resultados = []
for c in CARTAS:
    try:
        resultados.append(evaluate.evaluate_carta(c, cfg, suffix=SUFIXO))
        print('ok', c)
    except Exception as e:
        print('sem resultado para', c, '-', str(e)[:80])
resumo = evaluate.summary(resultados)
resumo.round(3)

## 4 · Área urbana ao longo do tempo (Col 12 × Col 11)

In [ ]:
import matplotlib.pyplot as plt
n = len(resultados)
fig, axs = plt.subplots((n + 2) // 3, 3, figsize=(14, 3.2 * ((n + 2) // 3)), squeeze=False)
for ax, r in zip(axs.flat, resultados):
    r['area'][['col12', 'col11']].plot(ax=ax, title=r['carta'])
    ax.set_ylabel('km²')
plt.tight_layout()

## 5 · Mapa de diferenças de uma carta

In [ ]:
import geemap
CARTA = CARTAS[0]; ANO = 2025
novo = evaluate.map_asset(CARTA, cfg, SUFIXO).select(f'classification_{ANO}')
velho = col11.final_map(CARTA).select(f'classification_{ANO}')
m = geemap.Map(); m.centerObject(grids.region(CARTA, cfg), 10)
m.addLayer(velho.selfMask(), {'palette': ['999999']}, 'Col 11', False)
m.addLayer(novo.selfMask(), {'palette': ['000000']}, 'Col 12', False)
m.addLayer(novo.subtract(velho).selfMask(), {'palette': ['red']}, 'só Col 12')
m.addLayer(velho.subtract(novo).selfMask(), {'palette': ['blue']}, 'só Col 11')
m

## 6 · Relatório HTML
Salva as tabelas num arquivo em `logs/relatorios/` (local, não vai para o GitHub).

In [ ]:
print(evaluate.html_report(resultados, cfg))